In [ ]:
import pandas as pd
import numpy as np
import warnings
import empyrical
import dai
import bigcharts 
import time 
warnings.filterwarnings('ignore')
print('导入包完成!')

In [ ]:
sql = """
with t1 as(
select
date as time,
date::DATE as date,
trading_day,
instrument,
volume,
sum(volume) over(partition by trading_day,instrument) as total_v,
volume/total_v as portion,
avg(portion) over(partition by date) as avg_portion,
abs(portion - avg_portion) as diff_portion,


from
cn_stock_bar1m_derived_c
order by time,instrument)
select
date,
instrument,
(sum(diff_portion*-1 * log(diff_portion+0.0000000001)))*-1 as factor
from
t1
group by date,instrument

"""

In [ ]:
def run(sql,shift_days):

    from bigquant import bigtrader, dai
    import pandas as pd
    from datetime import datetime, timedelta
    import numpy as np
    from sklearn.linear_model import LinearRegression

    def initialize(context: bigtrader.IContext):
        from bigtrader.finance.commission import PerOrder

        # 系统已经设置了默认的交易手续费和滑点，要修改手续费可使用如下函数
        context.set_commission(PerOrder(buy_cost=0.0003, sell_cost=0.0013, min_cost=5))


        context.holding_days = 5
        context.target_hold_count = 50


    def befor_trading(context, data):
        pass
        
    def handle_data(context: bigtrader.IContext, data: bigtrader.IBarData):

        # 每 context.holding_days 个交易日调仓一次
        if context.trading_day_index % context.holding_days != 0:
            return




        # 获取当前日期
        ed = data.current_dt.strftime("%Y-%m-%d")
        
        date_obj = datetime.strptime(ed, "%Y-%m-%d")
    
        # 向前推10天
        n_days_ago = date_obj - timedelta(days=shift_days)
        tomorrow = date_obj + timedelta(days=1)
    
        # 转换回字符串格式
        sd = n_days_ago.strftime("%Y-%m-%d")
        ed2 = tomorrow.strftime("%Y-%m-%d")
        
        # 获取当日数据
        current_day_data = dai.query(sql,filters={'date':[sd,ed2]}).df()
        current_day_data['date']=pd.to_datetime(current_day_data['date']) 
        current_day_data = current_day_data[current_day_data.date==ed]

        # 取前10只
        current_day_data.sort_values(by='factor',inplace=True,ascending=True)

        current_day_data = current_day_data.head(context.target_hold_count)
        len_ = len(current_day_data)
        # 获取当日目标持有股票
        target_hold_instruments = set(current_day_data["instrument"])
        
        # 获取当前已持有股票
        current_hold_instruments = set(context.get_account_positions().keys())

        # 卖出不在目标持有列表中的股票
        for instrument in current_hold_instruments - target_hold_instruments:
            context.order_target_percent(instrument, 0)
            
        # 买入目标持有列表中的股票
        for instrument in target_hold_instruments - current_hold_instruments:
            context.order_target_percent(instrument, 1/len_)

    performance = bigtrader.run(
        market=bigtrader.Market.CN_STOCK,
        frequency=bigtrader.Frequency.DAILY,
        start_date='2024-01-01',  
        end_date='2024-01-10',  
        capital_base=3000000,     # 设置初始资金
        initialize=initialize,     # 传入初始化函数
        handle_data=handle_data,   # 传入数据处理函数
        before_trading_start = befor_trading,
        order_price_field_buy='open',
        order_price_field_sell='open',
    )

    # 渲染绩效报告，展示回测结果
    performance.render()